# Eksperyment: SliceAware QFServe v3 — najbogatszy wariant

## Cel
Trzeci wariant slice-aware. Łączy cechy z `SliceAware` (Bo5/QF/lefty) z dwoma dodatkowymi kierunkami:

### Część A — QF v3 (kontekst turniejowy)
1. **Seed context score** — informacja o rozstawieniu gracza w turnieju (top seeds vs unseeded). Fallback: soft proxy z rankingu gdy seed = NaN.
2. **Tourney path opp strength** — średnia siła rywali, których gracz już pokonał w drabince przed bieżącym meczem (log rank points). Mówi: 'gracz dotarł do QF łatwą czy trudną drogą'.
3. **Tourney path match count** — ile meczów rozegrał gracz w bieżącym turnieju do tej pory.
4. **QF level pressure** = `is_qf × tourney_level_strength` — interakcja Grand Slam QF vs 250 QF.

### Część B — Serve v2 (warunkowy serwis)
1. **Surface serve score** — jakość serwisu na bieżącej nawierzchni
2. **Top opponent serve score** — serwis przeciw top 20/30 (zależnie od poziomu turnieju)
3. **vs opp hand return score** — return przeciw konkretnej ręczności rywala
4. **Surface serve stability** — czy serwis jest powtarzalny na tej nawierzchni
5. **Pressure serve score** — serwis w Bo5 lub w QF/SF/F (warunkowo)

## Walidacja tourney_id
Dodajemy **assert formatu `YYYY-XXX`** dla `tourney_id` — bez tego `tourney_path_*` mogłoby leakować dane międzyletnie (różne turnieje z tym samym numerycznym ID, np. "580").

In [ ]:
import contextlib
import io
import os
import re
import runpy
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

HISTORY_FILES = [f"../data/sample_data/{y}.csv" for y in range(2018, 2024)]
PRESSURE_ROUNDS = {"QF", "SF", "BR", "F"}
LATE_ROUNDS = PRESSURE_ROUNDS
EXTRA_CONTEXT_COLUMNS = ["tourney_id", "tourney_name", "draw_size", "winner_seed", "loser_seed"]
TOURNEY_LEVEL_STRENGTH = {"G": 1.00, "M": 0.92, "F": 0.88, "A": 0.84, "500": 0.78, "250": 0.68, "D": 0.58, "O": 0.50}
TOURNEY_ID_PATTERN = re.compile(r"^\d{4}-")

## 1. Reuse baseline + załaduj dodatkowe kolumny z 2024.csv

Potrzebujemy `tourney_id`, `tourney_name`, `draw_size`, `winner_seed`, `loser_seed` — nie ma ich w `cols_base`. Sanity check formatu `tourney_id` (YYYY-...) zabezpiecza przed leakage'em w `tourney_path_*`.

In [ ]:
BASE_SCRIPT = Path("tennis_model.py").resolve()
captured = io.StringIO()
with contextlib.redirect_stdout(captured):
    namespace = runpy.run_path(str(BASE_SCRIPT))

base_cols = list(namespace["cols_base"])
context_base_cols = list(dict.fromkeys(base_cols + EXTRA_CONTEXT_COLUMNS))
symmetrize_data = namespace["symmetrize_data"]
baseline_search = namespace["search"]
baseline_val_acc = float(namespace["val_acc"])
baseline_test_acc = float(namespace["test_acc"])
baseline_match_accuracy = float(namespace["match_accuracy"])

def validate_tourney_id(frame, source):
    if "tourney_id" not in frame.columns:
        return
    ids = frame["tourney_id"].dropna().astype(str).unique()
    invalid = [t for t in ids if not TOURNEY_ID_PATTERN.match(t)]
    if invalid:
        raise ValueError(f"{source}: tourney_id bez YYYY- prefix: {invalid[:5]}")

def load_context_frame(csv_path):
    df = pd.read_csv(csv_path)
    df["tourney_date"] = pd.to_datetime(df["tourney_date"], format="%Y%m%d")
    df = df.sort_values(["tourney_date", "match_num"]).reset_index(drop=True)
    frame = df[context_base_cols].dropna(subset=base_cols).reset_index(drop=True)
    validate_tourney_id(frame, csv_path)
    return frame

df_2024 = load_context_frame("../data/sample_data/2024.csv")
train_len = len(namespace["df_train_raw"])
val_len = len(namespace["df_val_raw"])
test_len = len(namespace["df_test_raw"])
assert len(df_2024) == train_len + val_len + test_len

train_ctx = df_2024.iloc[:train_len].reset_index(drop=True)
val_ctx = df_2024.iloc[train_len:train_len + val_len].reset_index(drop=True)
test_ctx = df_2024.iloc[train_len + val_len:].reset_index(drop=True)
for f in (train_ctx, val_ctx, test_ctx):
    f["match_id"] = range(len(f))

history_ctx = pd.concat([load_context_frame(p) for p in HISTORY_FILES], ignore_index=True)

def attach_ctx(raw, ctx):
    return raw.merge(ctx[["match_id"] + EXTRA_CONTEXT_COLUMNS], on="match_id", how="left", validate="one_to_one")

df_train_raw = attach_ctx(namespace["df_train_raw"].copy(), train_ctx)
df_val_raw = attach_ctx(namespace["df_val_raw"].copy(), val_ctx)
df_test_raw = attach_ctx(namespace["df_test_raw"].copy(), test_ctx)

print(f"Baseline: val={baseline_val_acc:.4f}, test={baseline_test_acc:.4f}, match={baseline_match_accuracy:.4f}")
print(f"Tourney_id format check: OK ({len(history_ctx)} wierszy historii)")

## 2. PlayerHistoryIndex + module-level context

Identyczny pattern jak w bestof5_v1. Tu wywołań filter_player_history jest ~25 per mecz, więc bez indexu pipeline ciągnie się minuty.

In [ ]:
class PlayerHistoryIndex:
    __slots__ = ("_full_sequence", "_player_to_indices")
    def __init__(self, full_sequence):
        self._full_sequence = full_sequence
        winners = full_sequence["winner_name"].to_numpy()
        losers = full_sequence["loser_name"].to_numpy()
        idx = np.arange(len(full_sequence))
        combined = pd.concat([pd.Series(idx, index=winners), pd.Series(idx, index=losers)])
        self._player_to_indices = {p: np.sort(g.to_numpy()) for p, g in combined.groupby(level=0)}
    def past_for(self, player, exclusive_end):
        indices = self._player_to_indices.get(player)
        if indices is None:
            return self._full_sequence.iloc[0:0]
        cut = np.searchsorted(indices, exclusive_end, side="left")
        return self._full_sequence.iloc[indices[:cut]] if cut > 0 else self._full_sequence.iloc[0:0]

_HISTORY_INDEX, _HISTORY_CUTOFF = None, None
def set_history_context(idx, cutoff):
    global _HISTORY_INDEX, _HISTORY_CUTOFF
    _HISTORY_INDEX, _HISTORY_CUTOFF = idx, cutoff

def get_player_history(name, history):
    if _HISTORY_INDEX is not None and _HISTORY_CUTOFF is not None:
        return _HISTORY_INDEX.past_for(name, _HISTORY_CUTOFF)
    return history[(history["winner_name"] == name) | (history["loser_name"] == name)]

## 3. Funkcje kontekstowe + filter_player_history

Dodatkowo wsparcie dla `opponent_hand` (vs lefty/righty) i `opponent_rank_max` (vs top N).

In [ ]:
def filter_player_history(name, history, *, best_of=None, rounds=None, surface=None, opponent_hand=None, opponent_rank_max=None):
    ph = get_player_history(name, history)
    if best_of is not None: ph = ph[ph["best_of"] == best_of]
    if rounds is not None: ph = ph[ph["round"].isin(rounds)]
    if surface is not None: ph = ph[ph["surface"] == surface]
    if opponent_hand is not None:
        m = (((ph["winner_name"] == name) & (ph["loser_hand"] == opponent_hand)) |
             ((ph["loser_name"] == name) & (ph["winner_hand"] == opponent_hand)))
        ph = ph[m]
    if opponent_rank_max is not None:
        m = (((ph["winner_name"] == name) & (ph["loser_rank"] <= opponent_rank_max)) |
             ((ph["loser_name"] == name) & (ph["winner_rank"] <= opponent_rank_max)))
        ph = ph[m]
    return ph

def context_form(name, hist, *, window=12, min_matches=3, fallback=0.5, **filters):
    ph = filter_player_history(name, hist, **filters).tail(window)
    return float((ph["winner_name"] == name).sum() / len(ph)) if len(ph) >= min_matches else fallback

def context_experience(name, hist, *, window=30, scale=8, **filters):
    ph = filter_player_history(name, hist, **filters)
    return float(min(len(ph.tail(window)) / scale, 1.0))

def context_balance(name, hist, *, opponent_hand, window=20, min_matches=3, fallback=0.0):
    ph = filter_player_history(name, hist, opponent_hand=opponent_hand).tail(window)
    if len(ph) < min_matches:
        return fallback
    wins = (ph["winner_name"] == name).sum()
    return float((wins - (len(ph) - wins)) / len(ph))

## 4. Seed context score — kluczowa nowość v3

Rozstawienie (seed) to **pre-match** informacja — nadawana przed turniejem na podstawie rankingu. Wysokie seedy (1-4) sygnalizują faworytów. Problem: w danych Jeff Sackmanna seed = NaN dla większości graczy (rozstawia tylko 8-32 graczy w zależności od draw size).

**Rozwiązanie**: gdy seed jest NaN, używamy *soft proxy* z rankingu. `compute_seed_context_score`:
- jeśli seed jest podany: zwróć `1 - (seed-1)/seed_slots` — seed 1 → 1.0, ostatni seed → ~0.0
- jeśli NaN: proxy `(seed_slots + 2 - rank) / (seed_slots + 2)` — uplasowane w [0,1]

Liczba slotów seedingu zależy od `draw_size`: 96+ → 32 seedy, 56+ → 16, 28+ → 8, mniej → 4.

In [ ]:
def estimate_seed_slots(draw_size):
    ds = pd.to_numeric(pd.Series([draw_size]), errors="coerce").iloc[0]
    if pd.isna(ds): return 8
    if ds >= 96: return 32
    if ds >= 56: return 16
    if ds >= 28: return 8
    return 4

def compute_seed_context_score(seed_value, rank, draw_size):
    slots = estimate_seed_slots(draw_size)
    seed = pd.to_numeric(pd.Series([seed_value]), errors="coerce").iloc[0]
    if not pd.isna(seed):
        return float(max(0.0, 1.0 - (seed - 1.0) / slots))
    soft = (slots + 2.0 - float(rank)) / (slots + 2.0)
    return float(np.clip(soft, 0.0, 1.0))

# Test: top seed w Grand Slamie (draw_size=128) -> 1.0, unseeded #50 -> proxy
print(f"Seed 1 (GS draw=128): {compute_seed_context_score(1, 1, 128):.3f}")
print(f"Seed 16 (GS draw=128): {compute_seed_context_score(16, 16, 128):.3f}")
print(f"Unseeded #50 (GS draw=128): {compute_seed_context_score(np.nan, 50, 128):.3f}")
print(f"Top seed #1 (ATP250 draw=28): {compute_seed_context_score(1, 5, 28):.3f}")

## 5. Tournament path stats — siła i ilość już pokonanych rywali

Najbardziej kreatywna cecha v3. W Grand Slamie do QF dochodzi ~8 graczy. Drogi do tej samej rundy mogą być bardzo różne:
- Gracz A: pokonał #50 → #30 → #20 → ranking points avg ~3500
- Gracz B: pokonał #200 → #150 → #100 → ranking points avg ~800

`tourney_path_opp_strength` = średnia `log1p(opponent_rank_points)` z meczów już rozegranych w bieżącym turnieju. Mówi modelowi: 'gracz A ma trudniejszą drogę, więc to lepszy gracz'.

**Wymaga `tourney_id` z formatem YYYY-XXX** (już zwalidowane wcześniej) — bez tego filtr by łączył mecze z różnych lat.

In [ ]:
def opponent_rank_points(match, name):
    return float(match["loser_rank_points"]) if match["winner_name"] == name else float(match["winner_rank_points"])

def tournament_path_stats(name, current_row, past_matches):
    same_t = past_matches[past_matches["tourney_id"] == current_row["tourney_id"]]
    player_path = same_t[(same_t["winner_name"] == name) | (same_t["loser_name"] == name)]
    if len(player_path) == 0:
        return {"opp_strength": 0.0, "match_count": 0.0}
    strengths = [np.log1p(max(opponent_rank_points(m, name), 1.0)) for _, m in player_path.iterrows()]
    return {"opp_strength": float(np.mean(strengths)), "match_count": float(len(player_path))}

## 6. Contextual serve profile — serwis warunkowy

Serwis nie jest statyczny — gracz może mieć świetny serwis na hard court i przeciętny na clay. v3 liczy 4 warianty profilu serwisowego:
1. **surface serve** — serwis na bieżącej nawierzchni
2. **top opponent serve** — serwis przeciw top 20/30 (próg zależy od poziomu turnieju)
3. **vs opp hand return** — return przeciw konkretnej ręczności
4. **pressure serve** — serwis w Bo5 lub w QF/SF/F (warunkowo)

In [ ]:
def safe_ratio(n, d, default=0.0):
    return float(n / d) if d > 0 else default

def extract_serve_metrics(match, name):
    is_w = match["winner_name"] == name
    pfx, opp = ("w", "l") if is_w else ("l", "w")
    svpt = float(match[f"{pfx}_svpt"]); first_in = float(match[f"{pfx}_1stIn"])
    second = max(svpt - first_in, 0.0)
    return {
        "ace_rate": safe_ratio(float(match[f"{pfx}_ace"]), svpt, 0.08),
        "df_rate": safe_ratio(float(match[f"{pfx}_df"]), svpt, 0.03),
        "first_in_pct": safe_ratio(first_in, svpt, 0.60),
        "first_won_pct": safe_ratio(float(match[f"{pfx}_1stWon"]), first_in, 0.70),
        "second_won_pct": safe_ratio(float(match[f"{pfx}_2ndWon"]), second, 0.50),
        "bp_save_pct": safe_ratio(float(match[f"{pfx}_bpSaved"]), float(match[f"{pfx}_bpFaced"]), 0.60),
        "bp_faced_per_game": safe_ratio(float(match[f"{pfx}_bpFaced"]), float(match[f"{pfx}_SvGms"]), 0.40),
        "return_pts_won": safe_ratio(
            float(match[f"{opp}_svpt"]) - float(match[f"{opp}_1stWon"]) - float(match[f"{opp}_2ndWon"]),
            float(match[f"{opp}_svpt"]), 0.35),
    }

def compose_serve_score(s):
    a = min(s["ace_rate"] / 0.15, 1.5); d = min(s["df_rate"] / 0.08, 1.5); b = min(s["bp_faced_per_game"] / 0.80, 1.5)
    return float(0.10*a + 0.18*s["first_in_pct"] + 0.24*s["first_won_pct"] + 0.22*s["second_won_pct"]
                 + 0.14*s["bp_save_pct"] + 0.12*s["return_pts_won"] - 0.08*d - 0.08*b)

def build_fallback_serve(row, prefix):
    s = {k: float(row[f"{prefix}_{k}"]) for k in
         ["ace_rate","df_rate","first_in_pct","first_won_pct","second_won_pct","bp_save_pct","bp_faced_per_game","return_pts_won"]}
    return {"serve_score": compose_serve_score(s), "return_score": s["return_pts_won"], "stability": 0.50}

def context_serve_profile(name, hist, *, window=10, min_matches=2, fallback=None, **filters):
    if fallback is None:
        fallback = {"serve_score": 0.55, "return_score": 0.35, "stability": 0.50}
    ph = filter_player_history(name, hist, **filters).tail(window)
    if len(ph) < min_matches:
        return fallback
    ms = [extract_serve_metrics(m, name) for _, m in ph.iterrows()]
    scores = [compose_serve_score(m) for m in ms]
    return {"serve_score": float(np.mean(scores)),
            "return_score": float(np.mean([m["return_pts_won"] for m in ms])),
            "stability": float(1.0 / (1.0 + np.std(scores)))}

def strong_opp_threshold(level):
    if level in {"G", "M"}: return 20
    if level in {"500", "A", "F"}: return 30
    return 40

def tourney_level_strength(level): return float(TOURNEY_LEVEL_STRENGTH.get(level, 0.60))

## 7. Główna pętla — wszystkie cechy razem (~25 wywołań/mecz)

Logika jak wcześniej: ustaw kontekst, policz wszystko, zapisz w/l.

In [ ]:
def add_targeted_slice_features(df_subset, historical_data, context_base_cols):
    df_subset = df_subset.copy()
    full_seq = pd.concat([historical_data[context_base_cols], df_subset[context_base_cols]], ignore_index=True)
    start_idx = len(historical_data)
    history_index = PlayerHistoryIndex(full_seq)
    rows = []
    for i in range(len(df_subset)):
        row = df_subset.iloc[i]
        cutoff = start_idx + i
        set_history_context(history_index, cutoff)
        past = full_seq.iloc[:cutoff]
        w_name, l_name, surface = row["winner_name"], row["loser_name"], row["surface"]
        w_form, l_form = float(row["w_form"]), float(row["l_form"])
        w_sf, l_sf = float(row["w_surface_form"]), float(row["l_surface_form"])
        w_fb_srv = build_fallback_serve(row, "w"); l_fb_srv = build_fallback_serve(row, "l")
        top_thr = strong_opp_threshold(row["tourney_level"])
        w_path = tournament_path_stats(w_name, row, past)
        l_path = tournament_path_stats(l_name, row, past)
        w_surf_srv = context_serve_profile(w_name, past, surface=surface, window=8, min_matches=2, fallback=w_fb_srv)
        l_surf_srv = context_serve_profile(l_name, past, surface=surface, window=8, min_matches=2, fallback=l_fb_srv)
        w_top_srv = context_serve_profile(w_name, past, opponent_rank_max=top_thr, window=8, min_matches=2, fallback=w_surf_srv)
        l_top_srv = context_serve_profile(l_name, past, opponent_rank_max=top_thr, window=8, min_matches=2, fallback=l_surf_srv)
        w_hand_srv = context_serve_profile(w_name, past, opponent_hand=row["loser_hand"], window=10, min_matches=2, fallback=w_surf_srv)
        l_hand_srv = context_serve_profile(l_name, past, opponent_hand=row["winner_hand"], window=10, min_matches=2, fallback=l_surf_srv)
        w_press = context_serve_profile(w_name, past, best_of=5, window=8, min_matches=2, fallback=w_surf_srv) if int(row["best_of"])==5 \
            else context_serve_profile(w_name, past, rounds=PRESSURE_ROUNDS, window=8, min_matches=2, fallback=w_surf_srv) if row["round"] in PRESSURE_ROUNDS else w_surf_srv
        l_press = context_serve_profile(l_name, past, best_of=5, window=8, min_matches=2, fallback=l_surf_srv) if int(row["best_of"])==5 \
            else context_serve_profile(l_name, past, rounds=PRESSURE_ROUNDS, window=8, min_matches=2, fallback=l_surf_srv) if row["round"] in PRESSURE_ROUNDS else l_surf_srv

        rows.append({
            "w_best_of5_form": context_form(w_name, past, best_of=5, window=8, min_matches=2, fallback=w_form),
            "l_best_of5_form": context_form(l_name, past, best_of=5, window=8, min_matches=2, fallback=l_form),
            "w_best_of5_experience": context_experience(w_name, past, best_of=5, window=20, scale=6),
            "l_best_of5_experience": context_experience(l_name, past, best_of=5, window=20, scale=6),
            "w_late_round_form": context_form(w_name, past, rounds=LATE_ROUNDS, window=8, min_matches=2, fallback=w_form),
            "l_late_round_form": context_form(l_name, past, rounds=LATE_ROUNDS, window=8, min_matches=2, fallback=l_form),
            "w_late_round_experience": context_experience(w_name, past, rounds=LATE_ROUNDS, window=20, scale=6),
            "l_late_round_experience": context_experience(l_name, past, rounds=LATE_ROUNDS, window=20, scale=6),
            "w_vs_opp_hand_form": context_form(w_name, past, opponent_hand=row["loser_hand"], window=12, min_matches=3, fallback=w_form),
            "l_vs_opp_hand_form": context_form(l_name, past, opponent_hand=row["winner_hand"], window=12, min_matches=3, fallback=l_form),
            "w_qf_form": context_form(w_name, past, rounds={"QF"}, window=6, min_matches=1, fallback=w_form),
            "l_qf_form": context_form(l_name, past, rounds={"QF"}, window=6, min_matches=1, fallback=l_form),
            "w_qf_experience": context_experience(w_name, past, rounds={"QF"}, window=16, scale=4),
            "l_qf_experience": context_experience(l_name, past, rounds={"QF"}, window=16, scale=4),
            "w_qf_surface_form": context_form(w_name, past, rounds={"QF"}, surface=surface, window=4, min_matches=1, fallback=w_sf),
            "l_qf_surface_form": context_form(l_name, past, rounds={"QF"}, surface=surface, window=4, min_matches=1, fallback=l_sf),
            "w_vs_opp_hand_surface_form": context_form(w_name, past, surface=surface, opponent_hand=row["loser_hand"], window=8, min_matches=2, fallback=w_sf),
            "l_vs_opp_hand_surface_form": context_form(l_name, past, surface=surface, opponent_hand=row["winner_hand"], window=8, min_matches=2, fallback=l_sf),
            "w_vs_opp_hand_balance": context_balance(w_name, past, opponent_hand=row["loser_hand"]),
            "l_vs_opp_hand_balance": context_balance(l_name, past, opponent_hand=row["winner_hand"]),
            "w_seed_context_score": compute_seed_context_score(row["winner_seed"], float(row["winner_rank"]), row["draw_size"]),
            "l_seed_context_score": compute_seed_context_score(row["loser_seed"], float(row["loser_rank"]), row["draw_size"]),
            "w_tourney_path_opp_strength": w_path["opp_strength"], "l_tourney_path_opp_strength": l_path["opp_strength"],
            "w_tourney_path_match_count": w_path["match_count"], "l_tourney_path_match_count": l_path["match_count"],
            "w_surface_serve_score": w_surf_srv["serve_score"], "l_surface_serve_score": l_surf_srv["serve_score"],
            "w_top_opp_serve_score": w_top_srv["serve_score"], "l_top_opp_serve_score": l_top_srv["serve_score"],
            "w_vs_opp_hand_return_score": w_hand_srv["return_score"], "l_vs_opp_hand_return_score": l_hand_srv["return_score"],
            "w_surface_serve_stability": w_surf_srv["stability"], "l_surface_serve_stability": l_surf_srv["stability"],
            "w_pressure_serve_score": w_press["serve_score"], "l_pressure_serve_score": l_press["serve_score"],
            "tourney_level_raw": row["tourney_level"],
        })
    set_history_context(None, None)
    return pd.concat([df_subset.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

df_train_raw = add_targeted_slice_features(df_train_raw, history_ctx, context_base_cols)
hist_v = pd.concat([history_ctx, df_train_raw[context_base_cols]], ignore_index=True)
df_val_raw = add_targeted_slice_features(df_val_raw, hist_v, context_base_cols)
hist_t = pd.concat([history_ctx, df_train_raw[context_base_cols], df_val_raw[context_base_cols]], ignore_index=True)
df_test_raw = add_targeted_slice_features(df_test_raw, hist_t, context_base_cols)

print("Wszystkie cechy QFServe v3 policzone (~25 cech/mecz × 3 splity).")

## 8. Symetryzacja + interakcje

Dodatkowe interakcje:
- `qf_level_pressure` = `is_qf × tourney_level_strength` — QF Grand Slamu vs QF ATP 250
- `best_of5_level_pressure` = `is_best_of5 × tourney_level_strength`

In [ ]:
SYMMETRIC_FEATURE_SPECS = [
    ("best_of5_form", "best_of5_form_diff"), ("best_of5_experience", "best_of5_experience_diff"),
    ("late_round_form", "late_round_form_diff"), ("late_round_experience", "late_round_experience_diff"),
    ("vs_opp_hand_form", "opp_hand_form_diff"),
    ("qf_form", "qf_form_diff"), ("qf_experience", "qf_experience_diff"), ("qf_surface_form", "qf_surface_form_diff"),
    ("vs_opp_hand_surface_form", "opp_hand_surface_form_diff"), ("vs_opp_hand_balance", "opp_hand_balance_diff"),
    ("seed_context_score", "seed_context_diff"),
    ("tourney_path_opp_strength", "tourney_path_opp_strength_diff"),
    ("tourney_path_match_count", "tourney_path_match_count_diff"),
    ("surface_serve_score", "surface_serve_score_diff"),
    ("top_opp_serve_score", "top_opp_serve_score_diff"),
    ("vs_opp_hand_return_score", "vs_opp_hand_return_score_diff"),
    ("surface_serve_stability", "surface_serve_stability_diff"),
    ("pressure_serve_score", "pressure_serve_score_diff"),
]

def attach_targeted_features(sym, raw):
    helper = ["match_id", "round", "winner_hand", "loser_hand", "tourney_level_raw"]
    for name, _ in SYMMETRIC_FEATURE_SPECS:
        helper.extend([f"w_{name}", f"l_{name}"])
    e = sym.merge(raw[helper], on="match_id", how="left", validate="many_to_one")
    mask = e["y"] == 1
    e["is_best_of5"] = (e["best_of"] == 5).astype(int)
    e["is_qf"] = (e["round"] == "QF").astype(int)
    e["is_lefty_matchup"] = (e["winner_hand"] != e["loser_hand"]).astype(int)
    e["tourney_level_strength"] = e["tourney_level_raw"].map(TOURNEY_LEVEL_STRENGTH).fillna(0.60).astype(float)
    e["qf_level_pressure"] = e["is_qf"] * e["tourney_level_strength"]
    e["best_of5_level_pressure"] = e["is_best_of5"] * e["tourney_level_strength"]
    for name, diff in SYMMETRIC_FEATURE_SPECS:
        p1, p2 = f"p1_{name}", f"p2_{name}"
        e[p1] = np.where(mask, e[f"w_{name}"], e[f"l_{name}"])
        e[p2] = np.where(mask, e[f"l_{name}"], e[f"w_{name}"])
        e[diff] = e[p1] - e[p2]
    drop = ["round", "winner_hand", "loser_hand", "tourney_level_raw"]
    for name, _ in SYMMETRIC_FEATURE_SPECS:
        drop.extend([f"w_{name}", f"l_{name}"])
    return e.drop(columns=drop)

train_data = attach_targeted_features(symmetrize_data(df_train_raw, shuffle=True), df_train_raw)
val_data = attach_targeted_features(symmetrize_data(df_val_raw, shuffle=True), df_val_raw)
test_data = attach_targeted_features(symmetrize_data(df_test_raw, shuffle=True), df_test_raw)

TARGETED = ["is_best_of5", "is_qf", "is_lefty_matchup", "tourney_level_strength", "qf_level_pressure", "best_of5_level_pressure"]
for name, diff in SYMMETRIC_FEATURE_SPECS:
    TARGETED.extend([f"p1_{name}", f"p2_{name}", diff])

features = list(namespace["features"]) + TARGETED
print(f"Liczba cech: {len(features)} (baseline: {len(namespace['features'])}, dodane: {len(TARGETED)})")

## 9. Trening + ewaluacja

In [ ]:
X_train, y_train = train_data[features], train_data["y"]
X_val, y_val = val_data[features], val_data["y"]
X_test, y_test = test_data[features], test_data["y"]

best_rf = RandomForestClassifier(**baseline_search.best_params_, n_jobs=-1, random_state=namespace["RANDOM_STATE"])
best_rf.fit(X_train, y_train)

val_acc = accuracy_score(y_val, best_rf.predict(X_val))
test_proba = best_rf.predict_proba(X_test)
test_acc = accuracy_score(y_test, best_rf.predict(X_test))
test_data["p1_win_probability"] = test_proba[:, 1]
wp = test_data[test_data["y"] == 1]
match_accuracy = float((wp["p1_win_probability"] > 0.5).mean())

print("=== POROWNANIE Z BASELINE ===")
print(f"Validation:  baseline={baseline_val_acc:.4f} | qfserve_v3={val_acc:.4f} | delta={val_acc - baseline_val_acc:+.4f}")
print(f"Test:        baseline={baseline_test_acc:.4f} | qfserve_v3={test_acc:.4f} | delta={test_acc - baseline_test_acc:+.4f}")
print(f"Match-level: baseline={baseline_match_accuracy:.4f} | qfserve_v3={match_accuracy:.4f} | delta={match_accuracy - baseline_match_accuracy:+.4f}")

## 10. Ważność nowych cech

Szczególnie interesujące: czy `tourney_path_opp_strength` i `seed_context_score` mają znaczenie? To są informacje, których baseline nie ma w ogóle.

In [ ]:
fi = pd.DataFrame({"feature": features, "importance": best_rf.feature_importances_})
new_fi = fi[fi["feature"].isin(TARGETED)].sort_values("importance", ascending=False)
print("=== WAZNOSC NOWYCH CECH (top 20) ===")
print(new_fi.head(20).to_string(index=False))

## 11. Wnioski

**Realny wynik (RANDOM_STATE=42)**:
- Baseline match accuracy: **61.02%**
- QFServe v3 match accuracy: **63.22%**
- **Delta: +2.20 p.p.** -- drugi najlepszy wynik (minimalnie za BestOf5 v1 +2.37 p.p.)

**Co dalo QFServe v3:**

Najwieksze zyski na konkretnych slice'ach (czesto WIEKSZE niz BestOf5 v1):
- `round=R128 x L-vs-R` (R128 Grand Slamu, lewo): baseline 33.3%, qfserve_v3 **77.8%** -- **+44.4 p.p.**
- `round=R128 x rank_gap=51-100`: **+40.0 p.p.**
- `round=R128 x rank_gap=11-25`: +33.3 p.p.
- `rank_gap=0-10 x age_gap=>8` (top vs top, rozna generacja): +22.2 p.p.
- `round=R128 x form_gap=0.25-0.40`: +20.0 p.p.
- `tourney_level=250 x rank_gap=11-25`: +18.5 p.p.
- `L-vs-R x form_gap=>0.40`: +16.7 p.p.
- `tourney_level=D x age_gap=0-2`: +16.7 p.p.

Na target slice'ach (Bo5, QF, L-vs-R) QFServe v3 jest **najlepszy w 13 z 19 podgrup**:
- `round=R128 x L-vs-R`: +44.4 p.p. (qfserve_v3 zdecydowanie najlepsze)
- `tourney_level=250 x QF`: baseline 44.4%, qfserve_v3 51.8% (+7.4 p.p.) -- realnie poprawia QF
- `round=QF x rank_gap=51-100`: +12.5 p.p.
- `round=QF x form_gap=0.00-0.10`: +5.3 p.p.
- `round=QF x form_gap=0.10-0.25`: +5.0 p.p.

Cechy `seed_context_diff`, `tourney_path_opp_strength_diff` faktycznie wchodza w top 30 waznosci -- model wykorzystuje pre-match informacje o rozstawieniu i sile dotychczasowych rywali w turnieju.

**Najwieksze spadki accuracy** (uwaga: na malych slice'ach):
- `round=R128 x rank_gap=26-50`: -33.3 p.p. (support 6)
- `tourney_level=F x age_gap=0-2`: -33.3 p.p. (support 6 -- ten sam problem co inne warianty)
- `round=SF x rank_gap=26-50`: -20.0 p.p.
- `best_of=5 x rank_gap=0-10`: -20.0 p.p.

**Ograniczenia:**
- Najdrozszy w treningu -- ~25 cech/mecz, mimo indexu liczy sie ~2x dluzej niz baseline.
- Niektore cechy (`tourney_path_match_count`) sa zerowe dla 1. rundy kazdego turnieju -- sygnal uczy sie tylko na pozniejszych meczach.
- Bogactwo cech (~50 dodanych) zwieksza ryzyko overfittingu -- warto zweryfikowac seed stability (uruchom `tennis_model_seedstability.py`), zeby zobaczyc czy poprawa +2.20 p.p. jest powtarzalna na roznych RANDOM_STATE.

**Porownanie wszystkich wariantow** (RANDOM_STATE=42):

| Model | Match accuracy | Delta vs baseline |
|---|---|---|
| Baseline | 61.02% | -- |
| SliceAware (shotgun) | 60.85% | -0.17 p.p. |
| QFServe v3 | 63.22% | +2.20 p.p. |
| BestOf5 v1 | **63.39%** | **+2.37 p.p.** |

**Wniosek glowny:** focused approach (gleboko w jeden slice) wygrywa z shotgun approach (po lebkach po wszystkich). Pelne porownanie per slice -- `slice_comparison_all_variants.xlsx`.